# 🚀 Модуль 4. Урок 3. Поиск аномалий и выбросов: Z-Score, метод Тьюки (IQR) и изоляционный лес

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_04_data_cleaning/lesson_03_outliers_anomaly_detection.ipynb)

## 📌 Бизнес-контекст и методология
Выбросы искажают средний чек и ломают метрики воронки. Существует два классических статистических метода:
1. **Правило 3 сигм (Z-Score)**: для нормальных распределений
2. **Метод межквартильного размаха (IQR)**: для распределений с тяжелыми хвостами

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd
import numpy as np

df_saas = pd.read_csv(saas_csv)

q1 = df_saas['mrr'].quantile(0.25)
q3 = df_saas['mrr'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df_saas[(df_saas['mrr'] < lower_bound) | (df_saas['mrr'] > upper_bound)]
print(f"Порог выбросов: < {lower_bound:.1f} или > {upper_bound:.1f}")
print(f"Обнаружено аномальных записей: {len(outliers)}")


Порог выбросов: < -76.0 или > 204.0
Обнаружено аномальных записей: 212


### 🎯 Практическое задание 1: Винзоризация (Winsorization)
Вместо удаления выбросов примените винзоризацию: ограничьте максимальные значения 99-м процентилем.

In [3]:
### ВАШ КОД ЗДЕСЬ
p99 = df_saas['mrr'].quantile(0.99)
df_winsorized = df_saas['mrr'].clip(upper=p99)

print(f"Исходный максимум: {df_saas['mrr'].max():.1f}")
print(f"Винзоризованный максимум: {df_winsorized.max():.1f}")

assert df_winsorized.max() <= p99, "Значения не должны превышать 99-й процентиль"
print("✅ Тест успешно пройден!")


Исходный максимум: 299.0
Винзоризованный максимум: 299.0
✅ Тест успешно пройден!


### 💼 Когда удалять выбросы нельзя?
В задачах поиска мошенничества (Fraud Detection) или оттока крупных клиентов именно выбросы содержат ключевую ценность.